<a href="https://colab.research.google.com/github/PurpleState/neural-tts/blob/main/hindi_tts_syspin_dataset_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══════════════════════════════════════
# MASTER INIT — run this every new session
# ═══════════════════════════════════════
from google.colab import drive
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

# Mount Drive
drive.mount('/content/drive')

# Paths
PROJECT       = '/content/drive/MyDrive/TTS_Hindi_Female'
BASE          = f"{PROJECT}/IISc_SYSPIN_Data/IISc_SYSPINProject_Hindi_Female_Spk001_HC"
WAV_DIR       = f"{BASE}/wav"
JSON_PATH     = f"{BASE}/IISc_SYSPINProject_Hindi_Female_Spk001_HC_Transcripts.json"
OUTPUT_DIR    = f"{PROJECT}/analysis_output"
PLOTS_DIR     = f"{OUTPUT_DIR}/plots"
RESAMPLED_DIR = f"{BASE}/wav_22k"
CSV_CACHE     = f"{OUTPUT_DIR}/df_full.csv"
TARGET_SR     = 22050
SPEAKER_NAME  = "hindi_female"

for d in [OUTPUT_DIR, PLOTS_DIR, RESAMPLED_DIR]:
    os.makedirs(d, exist_ok=True)

# save_plot helper
def save_plot(fig, name):
    path = f"{PLOTS_DIR}/{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved → {path}")

print("✓ Drive mounted")
print("✓ Paths set")
print("✓ Helpers ready")
print(f"\nChecking Drive...")
print(f"  WAV files : {len(os.listdir(WAV_DIR))} files")
print(f"  CSV cache : {'EXISTS ✓' if os.path.exists(CSV_CACHE) else 'NOT FOUND'}")

Mounted at /content/drive
✓ Drive mounted
✓ Paths set
✓ Helpers ready

Checking Drive...
  WAV files : 22058 files
  CSV cache : NOT FOUND


Build a subset of 500 samples

In [2]:
import json

# Load JSON
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

transcripts = data['Transcripts']

# Build subset — 500 samples, balanced across domains
rows = []
for file_id, meta in transcripts.items():
    rows.append({
        "file_id":      file_id,
        "text":         meta["Transcript"],
        "domain":       meta["Domain"],
        "audio_file":   os.path.join(WAV_DIR, f"{file_id}.wav"),
        "speaker_name": SPEAKER_NAME,
    })

full_df = pd.DataFrame(rows)
print(f"Full dataset: {len(full_df)} rows")
print(f"Domains: {full_df['domain'].value_counts().to_dict()}")

# Sample 50 from each domain (balanced)
subset_df = full_df.groupby('domain').apply(
    lambda x: x.sample(min(50, len(x)), random_state=42)
).reset_index(drop=True)

print(f"\nSubset size: {len(subset_df)} rows")
print(f"Subset domains:\n{subset_df['domain'].value_counts()}")

Full dataset: 22058 rows
Domains: {'BOOKS': 8358, 'OTHERS': 3081, 'GENERAL': 2540, 'EDUCATION': 2060, 'WEATHER': 1873, 'POLITICS': 1235, 'AGRICULTURE': 848, 'HEALTH': 835, 'FINANCE': 832, 'EVALUATION': 396}

Subset size: 500 rows
Subset domains:
domain
AGRICULTURE    50
BOOKS          50
EDUCATION      50
EVALUATION     50
FINANCE        50
GENERAL        50
HEALTH         50
OTHERS         50
POLITICS       50
WEATHER        50
Name: count, dtype: int64


/tmp/ipykernel_13550/4243468796.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  subset_df = full_df.groupby('domain').apply(


In [ ]:
subset_df = full_df.groupby('domain').apply(
    lambda x: x.sample(min(50, len(x)), random_state=42),
    include_groups=False
).reset_index(drop=True)

Compute durations on subset only

In [3]:
import librosa
from tqdm import tqdm

durations = []
for row in tqdm(subset_df.itertuples(), total=len(subset_df), unit="file"):
    try:
        d = librosa.get_duration(path=row.audio_file)
    except:
        d = 0.0
    durations.append(d)

subset_df = subset_df.copy()
subset_df["duration"]    = durations
subset_df["text_length"] = subset_df["text"].str.len()
subset_df["spc"]         = subset_df["duration"] / subset_df["text_length"].replace(0, np.nan)
spc_mean = subset_df["spc"].mean()
spc_std  = subset_df["spc"].std()

print(f"\n✓ Duration computed")
print(f"  Total duration  : {subset_df['duration'].sum()/3600:.2f} hours")
print(f"  Mean duration   : {subset_df['duration'].mean():.2f} s")
print(f"  Mean text length: {subset_df['text_length'].mean():.1f} chars")
print(f"  spc mean/std    : {spc_mean:.4f} / {spc_std:.4f}")

# Save immediately
subset_df.to_csv(CSV_CACHE, index=False)
print(f"\n✓ Saved to {CSV_CACHE}")

100%|██████████| 500/500 [05:12<00:00,  1.60file/s]


✓ Duration computed
  Total duration  : 1.12 hours
  Mean duration   : 8.03 s
  Mean text length: 92.2 chars
  spc mean/std    : 0.0875 / 0.0095

✓ Saved to /content/drive/MyDrive/TTS_Hindi_Female/analysis_output/df_full.csv


Plots

In [4]:
# ── Plot 1: Text length vs duration ──
subset_df["tl_bin"] = (subset_df["text_length"] // 10) * 10
grouped = subset_df.groupby("tl_bin")["duration"]
bins = sorted(subset_df["tl_bin"].unique())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Text length vs audio duration — Hindi Female SYSPIN (subset)")
axes[0].plot(bins, [grouped.get_group(b).mean()   for b in bins], marker="o", color="#7F77DD")
axes[0].set_title("Mean duration"); axes[0].set_xlabel("Text length (chars)")
axes[1].plot(bins, [grouped.get_group(b).median() for b in bins], marker="o", color="#1D9E75")
axes[1].set_title("Median duration"); axes[1].set_xlabel("Text length (chars)")
axes[2].plot(bins, [grouped.get_group(b).std()    for b in bins], marker="o", color="#D85A30")
axes[2].set_title("Std of duration"); axes[2].set_xlabel("Text length (chars)")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_plot(fig, "text_vs_duration")
plt.show()

# ── Plot 2: Distributions ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Dataset distributions — Hindi Female SYSPIN (subset)")
axes[0].hist(subset_df["duration"],    bins=30, color="#7F77DD", alpha=0.85)
axes[0].set_title("Audio duration (s)"); axes[0].set_xlabel("Seconds")
axes[1].hist(subset_df["text_length"], bins=30, color="#1D9E75", alpha=0.85)
axes[1].set_title("Text length (chars)"); axes[1].set_xlabel("Characters")
axes[2].hist(subset_df["spc"].dropna(),bins=30, color="#D85A30", alpha=0.85)
axes[2].axvline(spc_mean, color="black", linewidth=1.2,
                linestyle="--", label=f"mean={spc_mean:.3f}")
axes[2].set_title("Seconds per character")
axes[2].legend()
for ax in axes:
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_plot(fig, "distributions")
plt.show()

# ── Plot 3: Domain breakdown ──
domain_counts = subset_df["domain"].value_counts()
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(domain_counts.index, domain_counts.values, color="#534AB7", alpha=0.85)
ax.set_title("Samples per domain (subset)")
ax.set_xlabel("Domain"); ax.set_ylabel("Count")
ax.tick_params(axis='x', rotation=45)
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_plot(fig, "domain_breakdown")
plt.show()

# ── Plot 4: Word frequency ──
all_words = []
for text in subset_df["text"]:
    all_words.extend(text.strip().split())

word_freq = Counter(all_words)
print(f"Unique words : {len(word_freq)}")
print(f"Total words  : {sum(word_freq.values())}")

top50 = word_freq.most_common(50)
words, counts = zip(*top50)
fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(range(len(words)), counts, color="#7F77DD", alpha=0.85)
ax.set_xticks(range(len(words)))
ax.set_xticklabels(words, rotation=70, ha="right", fontsize=8)
ax.set_title("Top 50 most frequent words (subset)")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_plot(fig, "word_freq_top50")
plt.show()

  saved → /content/drive/MyDrive/TTS_Hindi_Female/analysis_output/plots/text_vs_duration.png
  saved → /content/drive/MyDrive/TTS_Hindi_Female/analysis_output/plots/distributions.png
  saved → /content/drive/MyDrive/TTS_Hindi_Female/analysis_output/plots/domain_breakdown.png
Unique words : 3777
Total words  : 8919


/tmp/ipykernel_13550/888999009.py:71: UserWarning: Glyph 2325 (\N{DEVANAGARI LETTER KA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_13550/888999009.py:71: UserWarning: Matplotlib currently does not support Devanagari natively.
  plt.tight_layout()
/tmp/ipykernel_13550/888999009.py:71: UserWarning: Glyph 2375 (\N{DEVANAGARI VOWEL SIGN E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_13550/888999009.py:71: UserWarning: Glyph 2350 (\N{DEVANAGARI LETTER MA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_13550/888999009.py:71: UserWarning: Glyph 2306 (\N{DEVANAGARI SIGN ANUSVARA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_13550/888999009.py:71: UserWarning: Glyph 2368 (\N{DEVANAGARI VOWEL SIGN II}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_13550/888999009.py:71: UserWarning: Glyph 2324 (\N{DEVANAGARI LETTER AU}) missing from font(s) DejaVu Sans.
  plt.tight_lay

  saved → /content/drive/MyDrive/TTS_Hindi_Female/analysis_output/plots/word_freq_top50.png


Fix Devanagari font and regenerate word frequency plot

In [5]:
# Install a font that supports Devanagari
!apt-get install -y fonts-noto
import matplotlib.font_manager as fm

# Refresh font cache and find Noto Sans Devanagari
fm.fontManager.__init__()
noto_fonts = [f for f in fm.findSystemFonts() if 'Noto' in f and 'Devanagari' in f]
print(f"Found Noto fonts: {noto_fonts}")

# Set it globally
if noto_fonts:
    noto_path = noto_fonts[0]
    prop = fm.FontProperties(fname=noto_path)
    print(f"Using: {noto_path}")
else:
    print("Noto Devanagari not found, trying fallback...")
    noto_path = None

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-noto-cjk fonts-noto-cjk-extra fonts-noto-color-emoji fonts-noto-core
  fonts-noto-extra fonts-noto-mono fonts-noto-ui-core fonts-noto-ui-extra
  fonts-noto-unhinted
The following NEW packages will be installed:
  fonts-noto fonts-noto-cjk fonts-noto-cjk-extra fonts-noto-color-emoji
  fonts-noto-core fonts-noto-extra fonts-noto-mono fonts-noto-ui-core
  fonts-noto-ui-extra fonts-noto-unhinted
0 upgraded, 10 newly installed, 0 to remove and 3 not upgraded.
Need to get 317 MB of archives.
After this operation, 790 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-core all 20201225-1build1 [12.2 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-noto all 20201225-1build1 [16.8 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-noto-cjk all 1:2

In [6]:
# Regenerate word frequency plot with correct font
top50 = word_freq.most_common(50)
words, counts = zip(*top50)

fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(range(len(words)), counts, color="#7F77DD", alpha=0.85)
ax.set_xticks(range(len(words)))

if noto_path:
    prop = fm.FontProperties(fname=noto_path, size=9)
    ax.set_xticklabels(words, rotation=70, ha="right", fontproperties=prop)
    ax.set_title("Top 50 most frequent words (subset)",
                 fontproperties=fm.FontProperties(fname=noto_path, size=13))
else:
    ax.set_xticklabels(words, rotation=70, ha="right", fontsize=9)
    ax.set_title("Top 50 most frequent words (subset)")

ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_plot(fig, "word_freq_top50")
plt.show()
print("✓ Word frequency plot regenerated with Devanagari support")

/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 108 (l) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 112 (p) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 84 (T) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 111 (o) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 109 (m) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 115 (s) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 116 (t) missing from font(s) Noto Sans Devanagari.
  plt.tight_layout()
/tmp/ipykernel_13550/1562034371.py:20: UserWarning: Glyph 102 (f) missing from font(s) Noto

  saved → /content/drive/MyDrive/TTS_Hindi_Female/analysis_output/plots/word_freq_top50.png
✓ Word frequency plot regenerated with Devanagari support


Final metadata csv

In [7]:
from sklearn.model_selection import train_test_split

# Flagged samples
z = (subset_df["spc"] - spc_mean) / spc_std
flagged = subset_df[z.abs() > 2.5].copy()
flagged["z_score"] = z[z.abs() > 2.5]
print(f"Flagged samples (|z|>2.5) : {len(flagged)}")
flagged.to_csv(f"{OUTPUT_DIR}/flagged_samples.csv", index=False)

# Too short
too_short = subset_df[subset_df["duration"] < 0.5]
print(f"Files under 0.5s          : {len(too_short)}")
too_short.to_csv(f"{OUTPUT_DIR}/too_short.csv", index=False)

# Train/test split
train_df, test_df = train_test_split(
    subset_df, test_size=0.10, random_state=42, shuffle=True
)
train_df[["audio_file","text","speaker_name","domain"]].to_csv(
    f"{OUTPUT_DIR}/metadata_train.csv", index=False)
test_df[["audio_file","text","speaker_name","domain"]].to_csv(
    f"{OUTPUT_DIR}/metadata_test.csv", index=False)

print(f"\nTrain : {len(train_df)} samples")
print(f"Test  : {len(test_df)} samples")

# Confirm all outputs
print("\n=== All saved files ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    full_path = f"{OUTPUT_DIR}/{f}"
    if os.path.isfile(full_path):
        size = os.path.getsize(full_path)/1024
        print(f"  {f:45s} {size:.1f} KB")
print("\nPlots:")
for f in sorted(os.listdir(PLOTS_DIR)):
    print(f"  {f}")

Flagged samples (|z|>2.5) : 8
Files under 0.5s          : 0

Train : 450 samples
Test  : 50 samples

=== All saved files ===
  df_full.csv                                   232.7 KB
  flagged_samples.csv                           3.7 KB
  metadata_test.csv                             19.3 KB
  metadata_train.csv                            179.1 KB
  too_short.csv                                 0.1 KB

Plots:
  distributions.png
  domain_breakdown.png
  text_vs_duration.png
  word_freq_top50.png
